#### 1.0 Libraries and directories

In [20]:
import ee 
import geemap
import geopandas as gpd
import pandas as pd
import datetime as dt
import pprint as pp
from shapely.geometry import shape

ee.Authenticate()
ee.Initialize(project='ee-green-by-another-name')

roi_name = 'YKF_sub1'
level = 'sr' # 1) 'sr': surface reflectance 2) 'toa' for top of atmosphere
resample_res = 30 
resample_method = 'bilinear'
n_dates = 1 # Number of observation dates (might be less if there aren't enough footprints)
s2_cloud_threshold = 20 # Cloud probability threshold for Sentinel-2 pixels. 
band_dict = {'Sentinel2_sr': ['B2', #Blue
                             'B3', #Green
                             'B4', #Red
                             'B8'], #NIR
             'LandSat8_sr': ['SR_B2', #Blue
                           'SR_B3', #Green
                           'SR_B4', #Red
                           'SR_B5'], #NIR
            'Sentinel2_toa': ['B2', 
                              'B3',
                              'B4',
                              'B8'],
            'LandSat8_toa': ['B2',
                             'B3',
                             'B4',
                             'B5']
}

image_footprints_path = f'./data/overlap_dates_for_roi/{roi_name}_overlap_dates.shp'
best_image_dates = gpd.read_file(image_footprints_path) 
est_utm = f'EPSG:{best_image_dates.estimate_utm_crs().to_epsg()}' # Have to convert pyproj object into literal string for ee


In [21]:
best_image_dates[['date', 'per_cover']].head(20)
print(best_image_dates.crs)

EPSG:4326


#### 2.0 Select target dates

In [22]:
best_image_dates['date_as_dt'] = pd.to_datetime(best_image_dates['date'])
# Incase I decide to analyze seasonality, filter into early and late date ranges
early = best_image_dates[best_image_dates['date_as_dt'].dt.month.isin([5, 6])]
early.head(10)
late = best_image_dates[best_image_dates['date_as_dt'].dt.month.isin([8, 9])]
late.head(10)

best_image_dates['date_plus_1d'] = best_image_dates['date_as_dt'] + pd.Timedelta(days=1)
#best_image_dates = best_image_dates[best_image_dates['date_as_dt'] > pd.to_datetime('2019-01-01')]
best_image_dates = best_image_dates[0:n_dates]

#print(best_image_dates[['date', 'per_cover']])

#### 3.0 Functions for the export pipeline

In [35]:
######################################
# Misc. helper functions
######################################
def convert_gpd_geom_to_ee(geom, est_utm):
        """
        Takes a geopandas geom object and coverts it to an Earth Engine polygon
        """
        if est_utm is None:
            out_crs = 'EPSG:4326'
        else:
            out_crs = est_utm

        coords = list(geom.exterior.coords)
        coords_list = [[x, y] for x, y in coords]
        return ee.Geometry.Polygon(coords_list, proj=out_crs)

def add_1d_to_date(date: str):
        date_plus_1d = pd.to_datetime(date) + pd.Timedelta(days=1)
        date_plus_1d.strftime('%Y-%m-%d') # Earth engine API needs strings
        return date_plus_1d

#########################################
# Part I: Functions to find the Sentinel-2 and Landsat8 image collections
#########################################

def find_col(polygon: ee.Geometry, 
             date: str, 
             level: str, 
             satellite: str):

    date_plus_1d = add_1d_to_date(date)
    
    if level == 'sr' and satellite == 'S2':
        asset_string = 'COPERNICUS/S2_SR_HARMONIZED'
    elif level == 'sr' and satellite == 'LS8':
         asset_string = 'LANDSAT/LC08/C02/T1_L2'
    elif level == 'toa' and satellite == 'LS8':
        asset_string = 'LANDSAT/LC08/C02/T1_TOA'
    elif level == 'toa' and satellite == 'S2':
        asset_string = 'COPERNICUS/S2_HARMONIZED'
    else:
        print(f'ERROR: level arg should be "sr" or "toa" not {level}')
    
    col = (
        ee.ImageCollection(asset_string)
        .filterDate(date, date_plus_1d)
        .filterBounds(polygon)
    )

    if col.size().getInfo() == 0:
        print(f'ERROR: No {satellite} images found for {date} with {asset_string} processing level')
        return None
    
    return col

#########################################
# Part II: Functions to select bands and rescale numerical values
#########################################

def fetch_rescale_imgs(s2_col: ee.ImageCollection,
                       polygon: ee.Geometry,
                       bands: list,
                       satellite: str):
    """
    Generates a single image mosiac with desired bands
    Rescales the bands to match (0-1) surface reflectance range
    TODO: Is rescaling different for TOA??
    """
        
    img = (s2_col.select(bands)
              .mosaic()
              .clip(polygon))
    
    def rescale_s2(img):
        rescaled_bands = img.divide(10_000)
        return rescaled_bands
    
    def rescale_ls8(img):
         rescaled_bands = img.multiply(0.0000275).add(-0.2)
         return rescaled_bands
    
    if satellite == 'S2':
         out_img = rescale_s2(img)
    elif satellite == 'LS8':
        out_img = rescale_ls8(img)
    else:
         print('ERROR: specify satellite as S2 or LS8')
    
    return out_img

#########################################
# Part III: Functions to produce individual cloud masks
#########################################
  
def make_s2_cloud_mask(polygon: ee.Geometry,
                       date: str, 
                       s2_col: ee.ImageCollection, 
                       s2_cloud_threshold: int,
                       level: str):
    """
    Produces a binary cloud mask from the Copernicus Cloud Probability 
    For SR data mask cirrus clouds and shaddows with the Sentinel-2 SCL (Scene Classification Layer)
    For TOA data use the Opaque and Cirrus cloud bands 
    """
    
    date_plus_1d = add_1d_to_date(date)

    s2_cloud_prob_string = 'COPERNICUS/S2_CLOUD_PROBABILITY'
    s2_clouds = (ee.ImageCollection(s2_cloud_prob_string)
                 .filterBounds(polygon)
                 .filterDate(date, date_plus_1d)
                 .mosaic()
                 .clip(polygon))
    
    #pp.pp(s2_clouds.bandNames().getInfo())

    clouds_binary = s2_clouds.select('probability').gt(s2_cloud_threshold).rename('cl_binary')
    
    # The SCL band is only available in the S2 Surface Reflectance Product!
    if level == 'sr':
        s2_scl = (s2_col.select('SCL')
                .mosaic()
                .clip(polygon))
        
        clouds_binary = s2_clouds.select('probability').gt(s2_cloud_threshold).rename('cl_binary')
        s2_shaddow_mask = s2_scl.eq(3)
        s2_cirrus_mask = s2_scl.eq(10) 
        s2_full_mask = clouds_binary.Or(s2_shaddow_mask).Or(s2_cirrus_mask)

    # Need to use different bands for TOA product, no SCL band available
    elif level == 'toa':
        s2_cirrus_mask = (s2_col.select('MSK_CLASSI_CIRRUS')
                          .mosaic()
                          .clip(polygon)
                          .eq(1)
        )

        s2_opaque_mask = (s2_col.select('MSK_CLASSI_OPAQUE')
                          .mosaic()
                          .clip(polygon)
                          .eq(1)
        )
        
        s2_full_mask = clouds_binary.Or(s2_opaque_mask).Or(s2_cirrus_mask)

    else:
        print('ERROR: specify level as "sr" or "toa"')

    return s2_full_mask

def make_ls8_cloud_mask(polygon: ee.Geometry, ls8_col: ee.ImageCollection):
    """
    Generates a cloud mask for Landsat 8 images using QA_PIXEL bit flags.
    """
    ls8_qa = (ls8_col
              .select('QA_PIXEL')
              .mosaic()
              .clip(polygon))
    
    # Define bitmasks for the conditions
    cloud_bit_mask = 1 << 3        # Bit 3: Cloud
    cloud_shadow_bit_mask = 1 << 4  # Bit 4: Cloud Shadow
    snow_bit_mask = 1 << 5         # Bit 5: Snow
    cirrus_bit_mask = 1 << 2       # Bit 2: Cirrus
    dilated_cloud_bit_mask = 1 << 1 # Bit 1: Dilated Cloud

    # Combine all bitmasks into one
    bitmask = (cloud_bit_mask
               | cloud_shadow_bit_mask
               | snow_bit_mask
               | cirrus_bit_mask
               | dilated_cloud_bit_mask)

    # Create the mask where any of the bits are set
    ls8_full_mask = ls8_qa.bitwiseAnd(bitmask).neq(0)

    return ls8_full_mask

def reduce_mask_resolution(mask: ee.Image, resample_res: int, est_utm: str):
    """
    Reduces the resolution of cloud masks
    Exports the reprojected cloud mask in UTM
    """

    original_crs = mask.projection()
    pp.pp(mask.projection().getInfo())
    mask_reproj = mask.reproject(
        crs=est_utm,
        scale=resample_res
    )
    mask_repoj_reduced = mask_reproj.reduceResolution(
        reducer=ee.Reducer.mean()
    ).reproject(
        crs=original_crs,
        scale=resample_res
    )

    return mask_repoj_reduced

#########################################
# Part IV: Functions to generate common masks and apply it to images
#########################################

def combine_sieve_dilate_masks(s2_mask, ls8_mask, est_utm, resample_res):
    size_threshold = 20  # Maximum number of pixels to ignore (sieve out) of the cloud mask
    dilation_radius = 1000  # The distance (meters) to dilate the clouds for a more conservative cloud mask.

    # Check that Landsat8 mask is local UTM, Sentinel-2 mask should already be reprojected
    # if str(ls8_mask.projection().getInfo()) is not est_utm:
    #     reproj_mask = ls8_mask.reproject(
    #         crs=ee.Projection(est_utm),
    #         scale=resample_res
    #     )
    #original_crs = s2_mask.projection()

    combined_mask = s2_mask.Or(ls8_mask)
    reproj_mask = combined_mask.reproject(
        crs=est_utm,
        scale=resample_res
    )
    # Sieve small clusters and dilate the cloud
    connected_pixels = reproj_mask.connectedPixelCount(maxSize=100, eightConnected=True)
    sieved_mask = reproj_mask.updateMask(connected_pixels.gte(size_threshold))
    dilation_kernel = ee.Kernel.circle(radius=dilation_radius, units='meters', normalize=False)
    dilated_mask = sieved_mask.focal_max(kernel=dilation_kernel, iterations=1)

    dilated_mask = dilated_mask.reproject(
        crs='EPSG:4326',
        scale=resample_res
    )

    return dilated_mask

def resample_img_then_mask(img: ee.Image, 
                           satellite: str,
                           common_mask: ee.Image, 
                           est_utm: str, 
                           resample_res: int,
                           resample_method: str):
    """
    Resample the image match the common cloud mask's resolution, then mask the image
    """
    original_proj = img.projection()
    #original_res = original_proj.nominalScale().getInfo()
    # Will always need to reduce the resolution for masking the Sentinel-2 image
    # Only reduce LS8 resolution if necessary. Unless resample_res > 30m, don't resample LS8
    if satellite == 'S2' or (satellite == 'LS8' and resample_res > 30):
        print(f'Resampling image for {satellite} at {resample_res}')
        reproj = img.reproject(
                crs=ee.Projection(est_utm),
                scale=resample_res
        )
        resamp = reproj.resample(resample_method).reproject(
            crs=original_proj,
            scale=resample_res
        )
        masked = resamp.updateMask(common_mask.neq(1))

    else:
        masked = img.updateMask(common_mask.neq(1))
    
    return masked

#########################################
# Part V: Main processsing & export functions
#########################################

def s2_processor(polygon: ee.Geometry, 
                date: str,
                level: str, 
                bands: list, 
                resample_res: int,
                est_utm: str, 
                s2_cloud_threshold: int):
    """
    Function to generate Sentinel-2 images and Mask
    """
    s2_col = find_col(polygon, date, level, satellite='S2')
    if s2_col is None:
        return None
    else: 
        s2_img = fetch_rescale_imgs(s2_col, polygon, bands, satellite='S2')
        s2_cloud_mask = make_s2_cloud_mask(polygon, date, s2_col, s2_cloud_threshold, level)
        s2_cloud_mask = reduce_mask_resolution(s2_cloud_mask, resample_res, est_utm)

        return s2_img, s2_cloud_mask
    
def ls8_processor(polygon: ee.Geometry, 
                  date: str,
                  level: str, 
                  bands: list,
                  resample_res: int,
                  est_utm):
    """
    Function to generate LandSat8 images and mask
    """
    ls8_col = find_col(polygon, date, level, satellite='LS8')
    if ls8_col is None:
         return None
    else:
        ls8_img = fetch_rescale_imgs(ls8_col, polygon, bands, satellite='LS8')
        ls8_cloud_mask = make_ls8_cloud_mask(polygon, ls8_col)
        # Don't bother resampling Landsat, if resample resolution = 30 meters
        if resample_res != 30:
            ls8_cloud_mask = reduce_mask_resolution(ls8_cloud_mask, resample_res, est_utm)
        else:
            return ls8_img, ls8_cloud_mask
    

def image_exporter(masked_s2: ee.Image, 
                   masked_ls8: ee.Image, 
                   polygon: ee.Geometry, 
                   footprint: gpd.GeoSeries,
                   resample_res: int,
                   resample_method: str,
                   level: str,
                   est_utm: str):
    """
    Exports the comonly masked images to Google Drive
    """
    if level == 'toa':
        folder = 'toa_images'
    elif level == 'sr':
        folder = 'sr_images'
    else:
        print('Error specify proper level')

    s2_export = ee.batch.Export.image.toDrive(
         image=masked_s2,
         description=f'Sentinel2-{level}_date_{footprint['date']}_roi_{roi_name}_resampled_{resample_method}{resample_res}',
         fileNamePrefix=f'Sentinel2-{level}_date_{footprint['date']}_roi_{roi_name}_resampled_{resample_method}{resample_res}',
         folder=folder,
         scale=30,
         region=polygon,
         crs='EPSG:4326',
         fileFormat='GeoTIFF',
         maxPixels=1e13
    )

    s2_export.start()
    print('Exporting Sentinel-2')

    ls8_export = ee.batch.Export.image.toDrive(
         image=masked_ls8,
         description=f'Landsat8-{level}_date_{footprint['date']}_roi_{roi_name}_resampled_{resample_method}{resample_res}',
         fileNamePrefix=f'Landsat8-{level}_date_{footprint['date']}_roi_{roi_name}_resampled_{resample_method}{resample_res}',
         folder=folder,
         scale=30,
         region=polygon,
         crs='EPSG:4326',
         fileFormat='GeoTIFF',
         maxPixels=1e13
    )

    ls8_export.start()
    print('Exporting Landsat8')

def main_processor(footprint: gpd.GeoSeries, 
                   level: str, 
                   band_dict: dict, 
                   resample_res: int, 
                   resample_method: str, 
                   est_utm: str, 
                   s2_cloud_threshold: int):

    polygon = convert_gpd_geom_to_ee(footprint['geometry'], est_utm=None)
    date = footprint['date']

    if level == 'toa':
        ls8_bands_key = 'LandSat8_toa'
        s2_bands_key = 'Sentinel2_toa'
    elif level == 'sr':
        ls8_bands_key = 'LandSat8_sr'
        s2_bands_key = 'Sentinel2_sr'
    else:
        print('Error specify proper level')
    
    s2_img, s2_cloud_mask = s2_processor(polygon=polygon,
                                        date=date,
                                        level=level,
                                        bands=band_dict[s2_bands_key],
                                        resample_res=resample_res,
                                        est_utm=est_utm,
                                        s2_cloud_threshold=s2_cloud_threshold)
    
    #pp.pp(s2_cloud_mask.projection().getInfo())
    
    ls8_img, ls8_cloud_mask = ls8_processor(polygon=polygon,
                                            date=date,
                                            level=level,
                                            bands=band_dict[ls8_bands_key],
                                            resample_res=resample_res,
                                            est_utm=est_utm)
    
    #pp.pp(ls8_cloud_mask.projection().getInfo())
    
    ####################
    ### Exports to test the alignment of not masked sentinel-2 and landsat8 images. 
    ####################

    # s2_test = ee.batch.Export.image.toDrive(
    #      image=s2_img,
    #      description=f'TESTING----Sentinel2-{level}_date_{footprint['date']}',
    #      fileNamePrefix=f'TESTING----Sentinel2-{level}_date_{footprint['date']}',
    #      folder='TEST',
    #      scale=30,
    #      region=polygon,
    #      crs='EPSG:4326',
    #      fileFormat='GeoTIFF',
    #      maxPixels=1e13
    # )

    # s2_test.start()
    # print('Exporting Sentinel-2')

    # ls8_test = ee.batch.Export.image.toDrive(
    #      image=ls8_img,
    #      description=f'TESTING----LANDSAT8-{level}_date_{footprint['date']}',
    #      fileNamePrefix=f'TESTING----LANDSAT8-{level}_date_{footprint['date']}',
    #      folder='TEST',
    #      scale=30,
    #      region=polygon,
    #      crs='EPSG:4326',
    #      fileFormat='GeoTIFF',
    #      maxPixels=1e13
    # )

    # ls8_test.start()
    # print('Exporting Landsat8')


    ######################################
    ######################################
    
    if s2_img is None or ls8_img is None:
         print(f'Matching images not sound on {date}')
    else:
        common_mask = combine_sieve_dilate_masks(s2_cloud_mask, ls8_cloud_mask, est_utm, resample_res)
        masked_s2 = resample_img_then_mask(s2_img, 'S2', common_mask, est_utm, resample_res, resample_method)
        masked_ls8 = resample_img_then_mask(ls8_img, 'LS8', common_mask, est_utm, resample_res, resample_method)
        # pp.pp(s2_img.projection().getInfo())
        # pp.pp(ls8_img.projection().getInfo())
        # pp.pp(masked_s2.projection().getInfo())
        # pp.pp(masked_ls8.projection().getInfo())
        image_exporter(masked_s2, masked_ls8, polygon, footprint, resample_res, resample_method, level, est_utm)


#### 4.0 Run the export pipeline for the ROI's date footprints and level

In [36]:
for idx, row in best_image_dates.iterrows():
    print(f'Processing {row['date']}')
    main_processor(
        footprint=row,
        level=level,
        band_dict=band_dict,
        resample_res=resample_res,
        resample_method=resample_method,
        est_utm=est_utm,
        s2_cloud_threshold=s2_cloud_threshold
    )

Processing 2019-05-16
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}
Resampling image for S2 at 30
Exporting Sentinel-2
Exporting Landsat8
